## seed_dim_fred_series
Seeds `silver.dim_fred_series` — FRED series metadata (label / units / native frequency) used by the Gold cadence-reconciliation views. Static reference data (10 rows). Idempotent MERGE on `series_id`. Wording from `_dev_planning/datasource_descriptions/fred_README.md`.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injects SILVER, F, spark. Static 10-series metadata (series_id, label, units, freq).
SERIES = [
    ("MORTGAGE30US",  "30-Year Fixed Rate Mortgage Average in the US",          "Percent",  "weekly"),
    ("MORTGAGE15US",  "15-Year Fixed Rate Mortgage Average in the US",          "Percent",  "weekly"),
    ("FIXHAI",        "Housing Affordability Index (Fixed)",                     "Index",    "monthly"),
    ("MSPUS",         "Median Sales Price of Houses Sold for the US",            "USD",      "quarterly"),
    ("UNRATE",        "Unemployment Rate",                                       "Percent",  "monthly"),
    ("MEHOINUSA672N", "Real Median Household Income in the US",                  "2022 USD", "annual"),
    ("ACTLISCOUUS",   "Realtor.com Active Listing Count, US",                    "Count",    "monthly"),
    ("CSUSHPISA",     "S&P/Case-Shiller US National Home Price Index, SA",       "Index",    "monthly"),
    ("CSUSHPINSA",    "S&P/Case-Shiller US National Home Price Index, NSA",      "Index",    "monthly"),
    ("CPIAUCSL",      "Consumer Price Index for All Urban Consumers, SA",        "Index",    "monthly"),
]
df = spark.createDataFrame(SERIES, "series_id string, series_label string, units string, frequency string")
df.createOrReplaceTempView("dim_fred_series_staging")

spark.sql(f"""
    MERGE INTO {SILVER}.dim_fred_series t USING dim_fred_series_staging s ON t.series_id = s.series_id
    WHEN MATCHED THEN UPDATE SET
        t.series_label=s.series_label, t.units=s.units, t.frequency=s.frequency,
        t.updated_ts=current_timestamp()
    WHEN NOT MATCHED THEN INSERT (series_id, series_label, units, frequency, inserted_ts, updated_ts)
        VALUES (s.series_id, s.series_label, s.units, s.frequency, current_timestamp(), current_timestamp())
""")
print(f"seed_dim_fred_series: dim_fred_series rows = {spark.table(f'{SILVER}.dim_fred_series').count()}")